# Easy Q4 - 台灣 2022 年各縣市年均地表溫度
使用 MODIS LST 資料，計算台灣各縣市 2022 年全年地表溫度均值（攝氏度），輸出 CSV。

In [ ]:
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project='0')

In [2]:
# 台灣縣市行政邊界 (FAO GAUL Level 2 for Taiwan)
taiwan = ee.FeatureCollection('FAO/GAUL/2015/level1') \
    .filter(ee.Filter.eq('ADM0_NAME', 'Taiwan'))

print('縣市數量:', taiwan.size().getInfo())

縣市數量: 1


In [3]:
# MODIS Terra LST Day 1km, 2022 全年
modis_lst = ee.ImageCollection('MODIS/061/MOD11A2') \
    .filterDate('2022-01-01', '2022-12-31') \
    .select('LST_Day_1km')

# LST_Day_1km 單位為 0.02 K，換算為攝氏度：T(°C) = DN * 0.02 - 273.15
def kelvin_to_celsius(image):
    return image.multiply(0.02).subtract(273.15) \
        .copyProperties(image, ['system:time_start'])

lst_celsius = modis_lst.map(kelvin_to_celsius)

# 全年平均
lst_annual_mean = lst_celsius.mean()

print('影像數量:', modis_lst.size().getInfo())

影像數量: 46


In [4]:
# 對每個縣市計算區域平均值
def reduce_region(feature):
    stats = lst_annual_mean.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=feature.geometry(),
        scale=1000,
        maxPixels=1e9
    )
    return feature.set('mean_lst_celsius', stats.get('LST_Day_1km'))

taiwan_with_lst = taiwan.map(reduce_region)

# 取出結果
result = taiwan_with_lst.select(['ADM1_NAME', 'mean_lst_celsius']).getInfo()
print('完成計算')

完成計算


In [5]:
# 整理成 DataFrame
rows = []
for feature in result['features']:
    props = feature['properties']
    rows.append({
        '縣市名稱': props.get('ADM1_NAME', ''),
        '年均地表溫度_攝氏': round(props.get('mean_lst_celsius', float('nan')), 2)
    })

df = pd.DataFrame(rows).sort_values('縣市名稱').reset_index(drop=True)
print(df.to_string(index=False))

        縣市名稱  年均地表溫度_攝氏
Taiwan Sheng      24.47


In [6]:
# 輸出 CSV
output_path = 'taiwan_lst_2022.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'已輸出至 {output_path}')

已輸出至 taiwan_lst_2022.csv
